# 调用具体模型厂商api
## 1.1 deepseek

In [3]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)
from langchain_deepseek import ChatDeepSeek

llm_deepseek = ChatDeepSeek(
    model="deepseek-v4-flash",
)

response = llm_deepseek.invoke("你好")
# 用 Markdown 渲染，保留模型返回的换行和列表格式
display(Markdown(response.content))

你好！很高兴见到你😊 有什么我可以帮你的吗？无论是聊天、解答问题，还是需要一些建议，我都会尽力支持你！

## 1.2 千问

In [4]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)
from langchain_qwq import ChatQwen

llm_chatqwen = ChatQwen(
    model="qwen3.5-ocr",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
)

response = llm_chatqwen.invoke("你好")
# 打印 response 对象会挤成一行，应取 content 并渲染
display(Markdown(response.content))

```json
{
    "text": "你好"
}
```

## 1.3兼容的写法
使用ChatOpenAI

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_openai import ChatOpenAI

load_dotenv(override=True)

llm_openai = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPEN_302_API_KEY"),
    base_url=os.getenv("OPEN_302_API_BASE"),
)

response = llm_openai.invoke("你好")
display(Markdown(response.content))


你好！有什么我可以帮助你的吗？

## 1.4 模型初始化方案 init_chat_model()

In [ ]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

llm = init_chat_model(
    model="gpt-4o-mini",
    api_key=os.getenv("OPEN_302_API_KEY"),
    base_url=os.getenv("OPEN_302_API_BASE"),
)

response = llm.invoke("你好")
display(Markdown(response.content))

你好！有什么我可以帮助你的吗？

## 1.5 本地模型

In [32]:
from langchain_ollama import ChatOllama

llm_ollama = ChatOllama(
    model="qwen2.5:3b",
    base_url="http://localhost:11434"
)

response = llm_ollama.invoke("你好")
display(Markdown(response.content))


你好！有什么问题可以随时和我交流。我是阿里云开发的预训练模型，目前可以回答一些知识性、技巧性和创意性的问题，并能根据您的指令生成文字。例如文章、剧本、小说章节等。您需要了解些什么吗？或者我们来聊聊天？

# 模型使用
## 模型初始化参数

以下参数适用于 `ChatOpenAI`、`ChatDeepSeek`、`ChatQwen`、`init_chat_model()` 等 LangChain Chat Model。

### 一、连接与认证（几乎所有厂商通用）

| 参数 | 类型 | 说明 | 示例 |
|------|------|------|------|
| `model` | `str` | 模型名称（必填） | `"gpt-4o-mini"`、`"deepseek-v4-flash"` |
| `api_key` | `str` | API 密钥，未传时通常从环境变量读取 | `os.getenv("OPENAI_API_KEY")` |
| `base_url` | `str` | 自定义 API 地址（代理/兼容接口/国内站） | `"https://dashscope.aliyuncs.com/compatible-mode/v1"` |
| `timeout` | `float` | 请求超时时间（秒） | `30` |
| `max_retries` | `int` | 失败重试次数 | `2` |
| `default_headers` | `dict` | 自定义 HTTP 请求头 | `{"Authorization": "Bearer xxx"}` |

In [30]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)
from langchain_qwq import ChatQwen

llm_chatqwen = ChatQwen(
    model="deepseek-v4-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    temperature=1
)

response = llm_chatqwen.invoke("你是谁")
# 打印 response 对象会挤成一行，应取 content 并渲染
display(response.content)

'你好！我是DeepSeek，由深度求索公司创造的AI助手。😊\n\n我是一个纯文本模型，可以帮你解答各种问题、进行对话、处理文档等。我支持：\n- 📝 阅读链接和处理上传的文件（图片、PDF、Word、Excel、PPT等）\n- 🌐 联网搜索（需要手动开启）\n- 💬 上下文对话，最多支持1M token的超长文本\n- 🎤 App端的语音输入\n\n目前我是完全免费的，没有任何收费计划。有什么我可以帮你的吗？无论是学习、工作还是日常问题，尽管问我！✨'

# 模型的调用
- invoke(): 阻塞式，一次性返回完整结果回答
- ainvoke(): 非阻塞，提高系统吞吐量高并发web应用、I/O密集型任务
- stream(): 流式传输，实时返回每个token,聊天机器热、长文本生成
- astream(): 非阻塞，提高系统吞吐量高并发web应用、I/O密集型任务
- batch(): 批量处理，适合需要处理大量请求的场景
- abatch(): 非阻塞，提高系统吞吐量高并发web应用、I/O密集型任务

> `from langchain_core.messages import HumanMessage, AIMessage, SystemMessage`,有3个对象可以用来创建消息。`HumanMessage(content="xxx")`

## 2.1 invoke()
`response=model.invoke(input,config=one)`

- input: 输入文本
  - 字符串
  - 列表[str]
  - 字典[{"role":"user","content":"你好"},{"role":"assistant","content":"你好"}]
- config: 配置参数(dict)
  - max_tokens: 最大token数
  - temperature: 温度
  - top_p: 上文概率
  - top_k: 上文数量
- response: 返回结果
  - content: 返回结果的content
  - additional_kwargs: 如果None是正常回答
  - response_metadata
  - id: 此次标识
  - tool_calls: 工具调用
  - invalid_tool_calls: 无效工具调用
  - usage_metadata: 使用情况
> - pretty_print(): 美化输出内容，但是只有content
> - rich库

In [38]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from rich import print as rprint

load_dotenv(override=True)
from langchain_qwq import ChatQwen

llm_chatqwen = ChatQwen(
    model="deepseek-v4-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    temperature=1
)

response = llm_chatqwen.invoke("你是谁")
# 打印 response 对象会挤成一行，应取 content 并渲染
rprint(response)

AIMessage(
    content='嗨！我是 **DeepSeek**，由深度求索公司创造的AI助手！😊\n\n简单介绍一下自己：\n- 
**免费使用**：目前完全免费，没有任何收费计划\n- **文本模型**：擅长对话、写作、编程、分析等各种文字任务\n- 
**超大上下文**：1M token容量，可以一次处理《三体》三部曲那么大的内容\n- 
**文件处理**：支持上传图片、PDF、Word、Excel、PPT等文件，从中读取文字信息\n- 
**联网搜索**：需要时可以手动开启联网功能获取最新信息\n- 
**语音输入**：App端支持语音交流\n\n我的知识截止到2025年5月，会尽力用热情、细腻的方式回答你的问题。有什么我可以帮你
的吗？无论学习、工作还是生活小事，都欢迎和我聊聊！💡',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 
'嗯，用户问了一个很基础的自我介绍问题：“你是谁”。这是一个非常常见且简单的问题，用户可能是第一次接触我，想了解我的基
本身份和功能。\n\n我需要给出清晰、简洁且友好的自我介绍。应该直接说明我是谁（DeepSeek），由哪家公司创造（深度求索）
，然后简要列举我的核心能力（文本模型、文件处理、联网搜索等）和关键特点（免费、长上下文、支持语音等）。最后以开放式
的邀请结束，询问用户是否需要帮助，这样既完成了自我介绍，又促进了进一步互动。\n\n想到了可以用热情的开头，然后分块但
不用序号地介绍主要特点，保持信息明确且易读。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 315,
            'prompt_tokens': 5,
            'total_tokens': 320,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 137,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
        },
        'model_provider': 'dashscope',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': None,
        'id': 'chatcmpl-49f73e65-08d6-9677-b3cb-53fd2b91d539',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f0200-3d4a-7332-9bcc-3dec81e68c24-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 5,
        'output_tokens': 315,
        'total_tokens': 320,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 137}
    }
)

## 2.2 stream() 流式调用
依赖于模型对流式输出的支持，会返回一个迭代器

In [39]:
# stream 流式生成
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from rich import print as rprint

load_dotenv(override=True)
from langchain_qwq import ChatQwen

llm_chatqwen = ChatQwen(
    model="deepseek-v4-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    temperature=1
)
for chunk in llm_chatqwen.stream("你是谁"):
    print(chunk.text,end="",flush=True)


1
你好！我是DeepSeek，由深度求索公司创造的AI助手。我是一个纯文本模型，可以帮你解答问题、处理文件、进行对话交流等。我支持上传图片、PDF、Word、Excel等文件，并能从中提取文字信息进行分析。我还支持联网搜索（需要手动开启），拥有1M的超长上下文，可以一次性处理大量文本。目前我是完全免费的，没有任何收费计划，你可以放心使用！有什么我可以帮你的吗？😊

In [40]:
# batch处理
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from rich import print as rprint

load_dotenv(override=True)
from langchain_qwq import ChatQwen

llm_chatqwen = ChatQwen(
    model="deepseek-v4-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
    temperature=1
)
result = llm_chatqwen.batch(['你是谁','你能做什么','现在是什么时候'])

rprint(result)

[
    AIMessage(
        content='你好！我是DeepSeek，由深度求索公司创造的AI助手。我是一个纯文本模型，可以帮你解答问题、处理信息、进
行对话等等。\n\n我的一些特点包括：\n- **免费使用**，目前没有任何收费计划\n- **知识截止日期**：2025年5月\n- 
**支持文件上传**：可以处理图像、txt、pdf、ppt、word、excel等文件，从中读取文字信息\n- 
**联网搜索功能**：需要手动开启\n- **上下文长度**：1M，可以一次性处理大量文本\n- 
**语音输入**：App端支持\n\n虽然我不支持多模态识别（比如直接“看懂”图片内容），但可以读取上传文件中的文字信息。有什么
我可以帮你的吗？无论是学习、工作还是日常问题，尽管问我！😊',
        additional_kwargs={
            'refusal': None,
            'reasoning_content': 
'好的，用户问“你是谁”，这是一个非常基础的自我介绍问题。用户可能是第一次接触我，想了解我的身份和基本功能。我需要直接
、清晰地说明我是谁、我的创造者、我的核心能力以及特点，让用户快速建立认知。想到了用友好的问候开头，然后明确我是DeepS
eek，由深度求索公司创造。接着列出关键信息：免费、知识截止时间、支持的功能（文件处理、联网搜索、长上下文等），并说明
文本模型和纯文本交互的特性，最后以开放式的帮助邀请结束。'
        },
        response_metadata={
            'token_usage': {
                'completion_tokens': 285,
                'prompt_tokens': 5,
                'total_tokens': 290,
                'completion_tokens_details': {
                    'accepted_prediction_tokens': None,
                    'audio_tokens': None,
                    'reasoning_tokens': 113,
                    'rejected_prediction_tokens': None
                },
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
            },
            'model_provider': 'dashscope',
            'model_name': 'deepseek-v4-flash',
            'system_fingerprint': None,
            'id': 'chatcmpl-6770cfed-b3fb-9727-a569-7821ea665bd8',
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='lc_run--019f028c-b5d7-76c3-b97b-b6f1a808f814-0',
        tool_calls=[],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 5,
            'output_tokens': 285,
            'total_tokens': 290,
            'input_token_details': {'cache_read': 0},
            'output_token_details': {'reasoning': 113}
        }
    ),
    AIMessage(
        content='你好呀！很高兴为你介绍我能做些什么！😊 我是DeepSeek，一个全能型AI助手，能帮你做很多事情：\n\n## 📚
**知识问答与学习**\n- 解答各种问题（科学、历史、文化、技术等）\n- 提供学习建议、解释复杂概念\n- 
辅导功课、批改作业\n\n## ✍️ **文本创作与处理**\n- 写文章、故事、诗歌、剧本\n- 润色、改写、翻译文本\n- 
总结长文档、提取要点\n- 写邮件、报告、演讲稿\n\n## 💻 **编程与技术**\n- 编写代码（Python、Java、JavaScript等）\n- 
调试程序、解释代码逻辑\n- 算法设计、数据结构分析\n\n## 🧠 **思考与分析**\n- 头脑风暴、创意策划\n- 
逻辑推理、问题拆解\n- 数据分析建议、决策辅助\n\n## 📄 **文件处理**（我的特色功能！）\n- 
处理上传的图片、PDF、Word、Excel、PPT、TXT文件\n- 从文件中提取文字信息进行分析\n- 
**特别提醒**：我不会识别图片内容，但能读取图片中的文字\n\n## 🌐 **联网搜索**（需手动开启）\n- 
在Web或App端点击联网搜索按钮\n- 查询最新信息、实时数据\n\n## 🗣️ **多语言交流**\n- 支持多种语言对话\n- 
可切换不同语气风格\n\n## 📱 **多端使用**\n- Web网页版\n- iOS/Android App（支持语音输入）\n- 可阅读链接内容\n\n## ⚠️
**我不能做的**\n- ❌ 生成图片、视频、音频\n- ❌ 实时更新知识（需联网搜索）\n- ❌ 执行代码或访问外部系统\n- ❌ 
主动发起对话或记忆个人信息\n\n有什么具体想让我帮忙的吗？无论是学习、工作还是生活问题，尽管问我！💪✨',
        additional_kwargs={
            'refusal': None,
            'reasoning_content': 
'好的，用户问“你能做什么”，这是一个非常常见且基础的问题。用户可能是第一次接触我，想了解我的功能范围，以便后续提出更
具体的问题。我需要给出一个全面但清晰的概述，涵盖主要能力领域，让用户快速建立认知。\n\n想到了按类别介绍，这样比较有
条理。可以从知识问答、文本处理、编程、学习辅助、文件处理等核心功能说起，再提一下我的特色比如多语言和长上下文。最后
需要说明一些限制，比如不能实时更新和生成图片，这样用户能合理预期。整体语气要友好热情，鼓励用户进一步提问。\n\n嗯，
回复结构可以这样：先热情打招呼，然后分几个大类列举我能做的事，再补充一些实用特色和限制，最后以邀请提问结束。避免过
于技术化的描述，用日常语言让用户容易理解。'
        },
        response_metadata={
            'token_usage': {
                'completion_tokens': 586,
                'prompt_tokens': 6,
                'total_tokens': 592,
                'completion_tokens_details': {
                    'accepted_prediction_tokens': None,
                    'audio_tokens': None,
                    'reasoning_tokens': 172,
                    'rejected_prediction_tokens': None
                },
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
            },
            'model_provider': 'dashscope',
            'model_name': 'deepseek-v4-flash',
            'system_fingerprint': None,
            'id': 'chatcmpl-08ae1d09-14fe-929f-852f-28de980cc33c',
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='lc_run--019f028c-b5d3-7462-a88f-784a9da07e20-0',
        tool_calls=[],
        invalid_tool_calls=[],
        u

## 异步方法
- 避免阻塞主线程
- 优化资源利用

In [ ]:

from langchain_qwq import ChatQwen
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
import asyncio
import time

load_dotenv(override=True)

model = ChatQwen(
    model="qwen3.5-ocr",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
)


async def demo_async_invoke():
    print("=== 演示：ainvoke 的异步（非阻塞）效果 ===")
    start_time = time.perf_counter()  # 记录开始时间

    print("程序开始...")

    # 1. 创建任务 (Task)
    print(">>> 发起异步模型调用 (ainvoke)...")
    async_task = asyncio.create_task(model.ainvoke("用一句话解释人工智能。"))

    # 2. 并行执行其他任务
    print(">>> 模型请求已在后台发送，继续执行本地逻辑...")
    for i in range(3):
        await asyncio.sleep(1)  # 使用异步等待，释放控制权
        print(
            f">>> 正在执行第{i + 1}个任务... (已耗时 {time.perf_counter() - start_time:.2f}s)")

    # 3. 获取模型结果
    print(">>> 本地任务完成，检查模型状态...")
    response = await async_task

    end_time = time.perf_counter()
    print(f">>> 模型返回: {response.content}")
    print(f"=== 总运行耗时: {end_time - start_time:.2f}s ===")


async def main():
    """主函数"""
    await demo_async_invoke()